# Dam removal effects over time, and what makes dams differ

**Questions:** (1) does the change after removal fade, grow or stay the same over the 5 years after it?
(2) are effects bigger or smaller at dams with particular characteristics (reservoir size, dam height, river size, land use, distance)?

**How to read every result**
- **Effect** = change after removal (log10 units for skewed measures: 0.1 = about +26%).
- **95% range** = plausible values for the effect. If it includes 0, we can't tell whether anything changed.
- **Verdict:** <span style="color:#c0392b">**detected**</span> (range excludes 0) · <span style="color:#27ae60">**no meaningful change**</span> (whole range inside the "too small to matter" threshold) · <span style="color:#7f8c8d">**inconclusive**</span> (can't tell).
- **Effect ÷ threshold:** charts divide by each measure's "too small to matter" threshold, so **±1 = the threshold** (e.g. 1 = 0.5 °C for temperature, a 25% change for turbidity).
- Results are associations, not proof of cause. Fewer than 15 dams = *exploratory*.

**Glossary**
- **BACI / before/after:** as in `distance_model_results.ipynb`: downstream change minus the change at upstream controls of the same dam (strong evidence), or downstream change only (supporting).
- **Year 1:** the first calendar year after the removal year. Removal dates are known only to the year, and the removal year itself is left out, so "year 1" is anywhere from 1 to 24 months after removal.
- **Time interval (bin):** a band of years after removal with its own effect: year 1, years 2–3, years 4–5.
- **Benchmark (binary) model:** the existing model with one effect for the whole 5 years, refit on the same data.
- **Effect modifier:** a characteristic that changes the size of the effect. It is *not* the same as a covariate: a covariate explains why water quality differs between places; a modifier explains why the *change after removal* differs between dams.
- **Per SD:** characteristics are measured in standard deviations (SD) across the dams, so "+0.5 per SD" = the effect is 0.5 larger at a dam one SD above the typical one.
- **q-value:** a p-value adjusted for testing many characteristics × measures at once. q < 0.1 = at most ~10% of such results are expected to be false leads.
- **AIC / BIC:** fit scores with a penalty for extra terms; **lower is better**. BIC penalizes extra terms more.

Run all cells; nothing is refit. Inputs come from `fit_distance_models.slurm` runs written to `stage5/`:
`RUNS="audit_10km audit_30km"`, `RUNS="time time_ml time_pretrend"`, `RUNS="modifiers modifiers_unadjusted"` and (optional) `COMBINED="…" RUNS=modifiers_combined`. Details: `docs/time_and_modifier_models.md`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap

# Find the project folder and the stage5 outputs written by the cluster jobs.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import distance_models as dm
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
STAGE5 = MODEL_DIR / "stage5"

def run_dir(run):
    """stage5/<run>__<random effects>, preferring auto (the reported runs)."""
    found = {p.name.split("__")[-1]: p for p in STAGE5.glob(f"{run}__*")}
    return next((found[r] for r in ("auto", "slopes", "intercepts") if r in found), None)

def read(run, name):
    folder = run_dir(run)
    path = folder / name if folder else None
    return pd.read_csv(path) if path and path.exists() and path.stat().st_size > 1 else pd.DataFrame()

def first(*frames):
    return next((f for f in frames if len(f)), pd.DataFrame())

# Time since removal (design 11).
time_results, time_at, time_cmp = read("time", "results.csv"), read("time", "effects_at.csv"), read("time", "time_comparison.csv")
ml_cmp = read("time_ml", "time_comparison.csv")
pre_results, pre_at = read("time_pretrend", "results.csv"), read("time_pretrend", "effects_at.csv")
cov_year = first(read("audit_10km", "time_coverage_by_year.csv"), read("time", "time_coverage_by_year.csv"))
cov_bin = first(read("audit_10km", "time_coverage_by_bin.csv"), read("time", "time_coverage_by_bin.csv"))
# Characteristics (design 12).
audit = first(read("audit_30km", "modifier_audit.csv"), read("modifiers", "modifier_audit.csv"))
corr = first(read("audit_30km", "modifier_correlations.csv"), read("modifiers", "modifier_correlations.csv"))
screen, screen_u = read("modifiers", "modifier_screen.csv"), read("modifiers_unadjusted", "modifier_screen.csv")
mod_at, mod_results = read("modifiers", "effects_at.csv"), read("modifiers", "results.csv")
comb_check, comb_coef = read("modifiers_combined", "combined_check.csv"), read("modifiers_combined", "coefficients.csv")
comb_results = read("modifiers_combined", "results.csv")

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
COLORS = {"detected": "#c0392b", "no meaningful change": "#27ae60", "inconclusive": "#7f8c8d"}
DESIGNS = {"baci": "BACI", "before_after": "before/after"}
MAIN = [s for s in rep.SESOI if s not in rep.DIAGNOSTIC_SERIES]   # discharge is a sampling check

def forest(ax, frame, est, lo, hi, label, color_col=None, title="", xlabel="effect ÷ threshold"):
    """One row per estimate: dot + 95% range, colored by verdict; grey band = ±1 threshold."""
    frame = frame.reset_index(drop=True)
    y = np.arange(len(frame))[::-1]
    colors = (frame[color_col].map(COLORS).fillna("#34495e") if color_col
              else pd.Series("#34495e", index=frame.index))
    ax.axvspan(-1, 1, color="0.9", lw=0); ax.axvline(0, color="black", lw=0.8)
    for yi, (_, r), color in zip(y, frame.iterrows(), colors):
        ax.plot([r[lo], r[hi]], [yi, yi], color=color, lw=1.5)       # 95% range
        ax.plot(r[est], yi, "o", color=color)                         # estimate
    ax.set_yticks(y, frame[label], fontsize=8); ax.set_title(title, fontsize=10); ax.set_xlabel(xlabel)

def per_threshold(frame, cols):
    """Divide the given columns by each row's threshold (±1 = too small to matter)."""
    frame = frame.copy()
    scale = frame["series"].map(rep.SESOI)
    for c in cols:
        frame[c] = frame[c] / scale
    return frame

def verdict(frame, est="estimate", lo="ci_low", hi="ci_high", threshold=True):
    return [rep.classify(e, l, h, rep.SESOI.get(s) if threshold else None)
            for e, l, h, s in zip(frame[est], frame[lo], frame[hi], frame["series"])]

for run in ("audit_10km", "audit_30km", "time", "time_ml", "time_pretrend", "modifiers",
            "modifiers_unadjusted", "modifiers_combined"):
    folder = run_dir(run)
    print(f"{run:<22} {folder.name if folder else 'not run yet'}")

## The models

**Time-varying BACI model.** The existing model has one "after removal" switch (*post*). Here it is split into intervals:

$$y = \beta_0 + \sum_k \gamma_k T_k + \beta_2\,\text{down} + \sum_k \delta_k\,\text{down}{\times}T_k + \text{month} + \text{weather} + u_{\text{dam}} + v_{\text{station}} + \varepsilon$$

- *T*<sub>k</sub> = 1 if the sample is in interval *k* (year 1, years 2–3, years 4–5); the 5 years before removal are the reference
- **δ<sub>k</sub> = the BACI effect in interval k.** Equal δ's = the effect doesn't change over time, and the model reduces to the benchmark (one effect, *post×down*)
- everything else as in the distance models; 10 km downstream, like the original BACI model

**Effect-modifier model** (one characteristic *m* at a time, measured in SDs, 0 = the typical dam):

$$y = \ldots + \beta_3\,\text{post}{\times}\text{down} + \beta_5\,\text{post}{\times}\text{down}{\times}m + (\text{distance and dilution terms}) + \ldots \qquad \text{effect} = \beta_3 + \beta_5\,m$$

- **β<sub>5</sub> = how much the effect changes per SD of the characteristic.** The model also lets *m* shift baselines (*m*, *post×m*, *down×m*), so β<sub>5</sub> isn't confused with places that are simply different.
- Dam characteristics (reservoir size, dam height, dam drainage area) have one value per dam. Land cover and distance belong to the **downstream** station (its upstream watershed), so they apply only to downstream stations.
- Distance and dilution stay in the model as **adjustments**, as in the main distance model; section 9 shows the results without them. 30 km downstream.
- Dams missing a characteristic are left out of that characteristic's model (never filled in), so each one is fit on its own set of dams.

## 1. Which time intervals can the data support?

To estimate a BACI effect in an interval, a dam needs downstream **and** upstream data in that interval. **Left:** how many dams have both, year by year (10 km). **Right:** the fewest such dams in any after-removal interval, for each way of grouping years; **blue = at least 10 dams in every interval** (the minimum to fit). The model uses **year 1 · years 2–3 · years 4–5**: year 1 on its own (when short-term disturbance like a sediment pulse would show), later years pooled to keep enough dams.

In [ ]:
if cov_year.empty:
    print("No coverage audit yet: RUNS=audit_10km sbatch data/slurm/fit_distance_models.slurm")
else:
    b = cov_year[cov_year["design"].eq("baci") & cov_year["series"].isin(MAIN)]
    grid = b.pivot_table(index="series", columns="years_from_removal", values="dams_both_roles").fillna(0)
    m = cov_bin[cov_bin["design"].eq("baci") & cov_bin["series"].isin(MAIN)].drop_duplicates(["series", "binning"])
    least = m.pivot_table(index="series", columns="binning", values="min_dams_post_bins").reindex(grid.index)
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.45 * len(grid) + 1.6), gridspec_kw={"width_ratios": [3, 2]})
    for ax, table, cmap, title in ((axes[0], grid, "Greys", "BACI dams with downstream AND upstream data, by year"),
                                   (axes[1], least, None, "fewest such dams in any after-removal interval")):
        if cmap:
            ax.imshow(table.to_numpy(dtype=float), cmap=cmap, aspect="auto", vmin=0, vmax=table.to_numpy().max() * 1.4)
        else:
            ax.imshow((table >= 10).astype(float).to_numpy(), cmap=ListedColormap(["#ecf0f1", "#aed6f1"]),
                      aspect="auto", vmin=0, vmax=1)
        for i in range(table.shape[0]):
            for j in range(table.shape[1]):
                if pd.notna(table.iat[i, j]):
                    ax.text(j, i, f"{int(table.iat[i, j])}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(table.shape[1]), [str(c) for c in table.columns], fontsize=8,
                      rotation=0 if cmap else 20)
        ax.set_yticks(range(table.shape[0]), table.index, fontsize=8); ax.set_title(title, fontsize=10)
    axes[0].set_xlabel("years from removal (removal year left out)")
    fig.tight_layout(); plt.show()

## 2. How does the effect change after removal?

One row per measure. **Dots and lines** = the effect in each interval with its 95% range, colored by verdict. **Blue line and band** = the benchmark: one effect for all 5 years, on the same data. **Hollow diamond** = the intervals averaged (weighted by samples), which should sit near the blue line. Numbers above the dots = dams with downstream data in that interval. Everything ÷ threshold (grey band = too small to matter).

What to look for: dots that **step down toward 0** = the change fades; **step away from 0** = it builds; **all near the blue line** = the single effect describes it well. Later intervals rest on fewer dams, so expect wider ranges there.

In [ ]:
if time_at.empty:
    print("No time models yet: RUNS=time sbatch data/slurm/fit_distance_models.slurm")
else:
    at = time_at[time_at["series"].isin(MAIN) & time_at["time_bin_period"].eq("post")].copy()
    at["design"] = at["spec"].str.replace("_time", "")
    at = per_threshold(at.assign(verdict=verdict(at)), ["estimate", "ci_low", "ci_high"])
    bench = time_results[time_results["spec"].str.endswith("_binary") & time_results["error"].fillna("").eq("")]
    bench = per_threshold(bench, ["effect_at_reference", "effect_ci_low", "effect_ci_high"]).set_index(["series", "design"])
    for series in [s for s in MAIN if s in set(at["series"])]:
        fig, axes = plt.subplots(1, 2, figsize=(12, 2.8))
        for ax, (design, label) in zip(axes, DESIGNS.items()):
            g = at[(at["series"] == series) & (at["design"] == design)]
            if g.empty:
                ax.set_visible(False); continue
            bins, avg = g[g["time_bin"] != "post average"], g[g["time_bin"] == "post average"]
            x = np.arange(len(bins))
            ax.axhspan(-1, 1, color="0.92", lw=0); ax.axhline(0, color="black", lw=0.8)
            if (series, design) in bench.index:
                b = bench.loc[(series, design)]
                ax.axhspan(b["effect_ci_low"], b["effect_ci_high"], color="#2980b9", alpha=0.15, lw=0)
                ax.axhline(b["effect_at_reference"], color="#2980b9", lw=1.2)
            for xi, (_, r) in zip(x, bins.iterrows()):
                color = COLORS.get(r["verdict"], "#34495e")
                ax.plot([xi, xi], [r["ci_low"], r["ci_high"]], color=color, lw=1.8)
                ax.plot(xi, r["estimate"], "o", color=color)
                ax.annotate(f"{int(r['treated_dams_in_bin'])}", (xi, r["ci_high"]), textcoords="offset points",
                            xytext=(0, 3), ha="center", fontsize=7, color="0.4")
            if len(avg):
                ax.plot(len(bins), avg["estimate"].iloc[0], "D", mfc="none", color="#34495e")
            ax.set_xticks(list(x) + [len(bins)], [b.replace("_", "–").replace("y", "yr ") for b in bins["time_bin"]] + ["avg"])
            ax.set_title(f"{series}: {label}", fontsize=10); ax.set_ylabel("effect ÷ threshold")
        fig.tight_layout(); plt.show()

## 3. Is one effect enough, or does it change over time?

Two ways to ask the same question, per measure:
- **change p** (Wald test): can we tell the intervals' effects apart? Small (< 0.05) = they differ.
- **ΔAIC / ΔBIC** = time model − one-effect model, from a separate fit made for comparing models (maximum likelihood). **Below −2 = the time model fits clearly better**; above +2 = the extra intervals aren't worth it. BIC penalizes extra terms more, so it favors the simpler model more often.

**Range width ratio** = width of the averaged time effect's 95% range ÷ the one-effect model's: about 1 means splitting by time costs no precision for the overall answer. A note appears where a fit didn't reach its best solution; that row's comparison is left blank.

In [ ]:
if time_cmp.empty:
    print("No time comparison yet: RUNS=time sbatch data/slurm/fit_distance_models.slurm")
else:
    t = time_cmp[time_cmp["series"].isin(MAIN)].copy()
    t["design"] = t["design"].map(DESIGNS)
    t["range width ratio"] = t["time_average_ci_width"] / t["binary_ci_width"]
    if len(ml_cmp):
        keep = [c for c in ("series", "design", "lrt_p", "delta_aic_time_minus_binary",
                            "delta_bic_time_minus_binary", "lrt_note") if c in ml_cmp]
        ml = ml_cmp[keep].assign(design=ml_cmp["design"].map(DESIGNS))
        t = t.drop(columns=[c for c in ("lrt_p", "lrt_note") if c in t]).merge(ml, on=["series", "design"], how="left")
    cols = {"series": "series", "design": "design", "dams": "dams", "binary_effect": "one effect",
            "time_average_effect": "time model, averaged", "range width ratio": "range width ratio",
            "time_change_p": "change p", "lrt_p": "LRT p", "delta_aic_time_minus_binary": "ΔAIC",
            "delta_bic_time_minus_binary": "ΔBIC", "lrt_note": "note"}
    display(t[[c for c in cols if c in t]].rename(columns=cols).sort_values(["design", "series"]).fillna(""))
    if "delta_aic_time_minus_binary" in t:
        c = t.dropna(subset=["delta_aic_time_minus_binary"]).copy()
        c["label"] = c["series"] + " (" + c["design"] + ", " + c["dams"].astype(int).astype(str) + " dams)"
        c = c.sort_values("delta_aic_time_minus_binary")
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(c) + 1))
        ax.axvspan(-2, 2, color="0.9", lw=0); ax.axvline(0, color="black", lw=0.8)
        ax.barh(c["label"], c["delta_aic_time_minus_binary"],
                color=np.where(c["delta_aic_time_minus_binary"] < 0, "#2980b9", "#e67e22"))
        ax.set_xlabel("ΔAIC  (← changing over time fits better | one effect fits better →)")
        ax.tick_params(axis="y", labelsize=8); fig.tight_layout(); plt.show()

## 4. Were downstream and upstream already drifting apart before removal?

BACI assumes downstream and upstream would have moved together without the removal. A check: split the 5 years **before** removal into early (5–3 years before) and late (2–1 years before) and estimate a BACI "effect" between them. Nothing was removed then, so it **should be about 0**. A clear non-zero value (red) means the two sides were already diverging, and that measure's BACI effect may partly reflect that trend rather than the removal.

In [ ]:
pre = pre_at[pre_at["series"].isin(MAIN) & pre_at["time_bin_period"].eq("pre")] if len(pre_at) else pre_at
if pre.empty:
    print("No pre-trend check yet: RUNS=time_pretrend sbatch data/slurm/fit_distance_models.slurm")
else:
    p = pre.merge(pre_results[["series", "spec", "pretrend_p"]], on=["series", "spec"], how="left")
    p = per_threshold(p.assign(verdict=verdict(p, threshold=False)), ["estimate", "ci_low", "ci_high"])
    p["label"] = p["series"] + " (" + p["dams"].astype(int).astype(str) + " dams, p = " + p["pretrend_p"].map(lambda v: f"{v:.2f}") + ")"
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(p) + 1))
    forest(ax, p, "estimate", "ci_low", "ci_high", "label", "verdict",
           title="BACI 'effect' early vs late before removal (should be ~0)")
    fig.tight_layout(); plt.show()

## 5. Which characteristics can we test?

Share of the **BACI dams** that have each characteristic, per measure (number = dams with a value). A characteristic missing for many dams is tested on a smaller, possibly unrepresentative set. Mapped reservoir area counts only reservoirs mapped by the removal year. **Stored sediment isn't available in any of our sources**; the reservoir-size index is the closest stand-in.

In [ ]:
if audit.empty:
    print("No characteristic audit yet: RUNS=audit_30km sbatch data/slurm/fit_distance_models.slurm")
else:
    a = audit[audit["design"].eq("baci") & audit["series"].isin(MAIN) & audit["available"].fillna(False).astype(bool)]
    share = a.pivot_table(index="series", columns="characteristic", values="dams_with_value_%")
    count = a.pivot_table(index="series", columns="characteristic", values="dams_with_value")
    dams = a.groupby("series")["dams"].first()
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(share) + 1.4))
    ax.imshow(share.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
    for i in range(share.shape[0]):
        for j in range(share.shape[1]):
            if pd.notna(count.iat[i, j]):
                ax.text(j, i, f"{int(count.iat[i, j])}", ha="center", va="center", fontsize=8,
                        color="white" if share.iat[i, j] > 60 else "black")
    ax.set_xticks(range(share.shape[1]), share.columns, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(share.shape[0]), [f"{s} ({int(dams[s])})" for s in share.index], fontsize=8)
    ax.set_title("BACI dams with a value (color = %, number = dams; in brackets: all BACI dams)", fontsize=10)
    fig.tight_layout(); plt.show()
    spread = audit[audit["design"].eq("baci") & audit["series"].eq("water_temp")]
    display(spread[["characteristic", "label", "dams_with_value", "p10", "median", "p90"]]
            .rename(columns={"p10": "10th percentile", "p90": "90th percentile"}) if len(spread) else spread)
    display(audit[~audit["available"].fillna(False).astype(bool)][["characteristic", "note"]].drop_duplicates())

## 6. Do the characteristics travel together?

Rank correlation (Spearman ρ) between characteristics across the BACI dams. **|ρ| above 0.7 (dark)** = the two mostly move together, so the data can't tell which one matters; a result for one may really be about the other (or about something neither measures). Land-cover shares are linked by construction (more forest leaves less room for farms).

In [ ]:
if corr.empty:
    print("No correlations yet (audit_30km run).")
else:
    c = corr[corr["design"].eq("baci")]
    names = sorted(set(c["a"]) | set(c["b"]))
    rho = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
    for _, r in c.iterrows():
        rho.loc[r["a"], r["b"]] = rho.loc[r["b"], r["a"]] = r["spearman_rho"]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(rho.to_numpy(dtype=float), cmap="RdBu_r", vmin=-1, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and pd.notna(rho.iat[i, j]):
                ax.text(j, i, f"{rho.iat[i, j]:.1f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(len(names)), names, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(names)), names, fontsize=8); fig.colorbar(im, ax=ax, label="Spearman ρ")
    fig.tight_layout(); plt.show()

## 7. Which characteristics go with bigger or smaller effects?

Each cell = one model: how much the effect changes **per SD** of the characteristic (÷ threshold). **Colored** = the 95% range excludes 0 (red = larger effect at dams higher in that characteristic, blue = smaller); **grey** = can't tell. **\*** = still stands after allowing for the many tests (q < 0.1). With ~25 BACI dams per measure, expect most cells grey, and treat a single colored cell without a \* as a lead, not a finding.

In [ ]:
if screen.empty:
    print("No screen yet: RUNS=modifiers sbatch data/slurm/fit_distance_models.slurm")
else:
    s = screen[screen["series"].isin(MAIN) & screen["error"].fillna("").eq("")].copy()
    s = per_threshold(s, ["modifier_slope", "modifier_ci_low", "modifier_ci_high"])
    s["detected"] = (s["modifier_ci_low"] > 0) | (s["modifier_ci_high"] < 0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 0.45 * s["series"].nunique() + 1.8))
    for ax, (design, label) in zip(axes, DESIGNS.items()):
        g = s[s["design"] == design]
        if g.empty:
            ax.set_visible(False); continue
        val = g.pivot_table(index="series", columns="characteristic", values="modifier_slope")
        hit = g.pivot_table(index="series", columns="characteristic", values="detected", aggfunc="first")
        q = g.pivot_table(index="series", columns="characteristic", values="q_value")
        lim = max(np.nanmax(np.abs(val.to_numpy(dtype=float))), 0.5)
        cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("0.9")
        ax.imshow(val.where(hit.fillna(False).astype(bool)).to_numpy(dtype=float), cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
        for i in range(val.shape[0]):
            for j in range(val.shape[1]):
                if pd.notna(val.iat[i, j]):
                    star = "*" if pd.notna(q.iat[i, j]) and q.iat[i, j] < 0.1 else ""
                    dark = hit.iat[i, j] is True and abs(val.iat[i, j]) > 0.6 * lim
                    ax.text(j, i, f"{val.iat[i, j]:+.2f}{star}", ha="center", va="center", fontsize=7,
                            color="white" if dark else "black")
        ax.set_xticks(range(val.shape[1]), val.columns, rotation=30, ha="right", fontsize=8)
        ax.set_yticks(range(val.shape[0]), val.index, fontsize=8)
        ax.set_title(f"{label}: change in effect per SD ÷ threshold", fontsize=10)
    fig.tight_layout(); plt.show()
    cols = ["design", "series", "characteristic", "dams", "dams_before_modifier", "modifier_slope",
            "modifier_ci_low", "modifier_ci_high", "q_value", "modifier_scale"]
    leads = s[s["detected"]].sort_values("q_value")
    print("Colored cells (range excludes 0), slope ÷ threshold; dams_before_modifier = dams before dropping missing values:")
    display(leads[[c for c in cols if c in leads]] if len(leads) else pd.DataFrame({"note": ["none"]}))

## 8. What does a modification look like in practice?

For the strongest BACI results above: the effect at a **low (10th percentile), typical (median) and high (90th percentile)** dam for that characteristic, with the actual values on the axis. A slope that only moves the effect within the grey band matters little in practice, even if it is "detected".

In [ ]:
if screen.empty or mod_at.empty:
    print("No screen yet.")
else:
    b = s[s["design"].eq("baci")].sort_values("modifier_p_value")
    top = b[b["q_value"] < 0.1].head(6)
    if top.empty:
        top = b.head(4)
        print("No BACI result passes q < 0.1; showing the 4 strongest for illustration only.")
    pts = mod_at[mod_at["characteristic_percentile"].notna()]
    fig, axes = plt.subplots(1, len(top), figsize=(3.3 * len(top), 3), squeeze=False)
    for ax, (_, r) in zip(axes[0], top.iterrows()):
        g = pts[(pts["series"] == r["series"]) & (pts["spec"] == r["spec"])].sort_values("characteristic_percentile")
        g = per_threshold(g.assign(verdict=verdict(g)), ["estimate", "ci_low", "ci_high"])
        ax.axhspan(-1, 1, color="0.92", lw=0); ax.axhline(0, color="black", lw=0.8)
        for xi, (_, p) in enumerate(g.iterrows()):
            color = COLORS.get(p["verdict"], "#34495e")
            ax.plot([xi, xi], [p["ci_low"], p["ci_high"]], color=color, lw=1.8); ax.plot(xi, p["estimate"], "o", color=color)
        ax.set_xticks(range(len(g)), [f"p{int(p)}\n{v:.3g}" for p, v in zip(g["characteristic_percentile"], g["characteristic_value"])], fontsize=8)
        ax.set_title(f"{r['series']}\nby {r['characteristic']}", fontsize=9); ax.set_ylabel("effect ÷ threshold")
    fig.tight_layout(); plt.show()

## 9. Does it matter that distance and dilution are held fixed?

Each dot = one measure × characteristic: the change per SD **with** distance and dilution in the model (across) vs **without** (up). Dots on the dashed line = the result doesn't depend on that choice. Dots far off it = part of the "characteristic" result was really about how far downstream, or how diluted, the stations were (e.g. land cover changes with distance).

In [ ]:
if screen.empty or screen_u.empty:
    print("Needs both RUNS=modifiers and RUNS=modifiers_unadjusted.")
else:
    key = ["series", "design", "characteristic"]
    both = per_threshold(screen, ["modifier_slope"]).merge(
        per_threshold(screen_u, ["modifier_slope"])[key + ["modifier_slope"]], on=key, suffixes=("", "_u"))
    both = both[both["series"].isin(MAIN)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    for ax, (design, label) in zip(axes, DESIGNS.items()):
        g = both[both["design"] == design]
        if g.empty:
            ax.set_visible(False); continue
        for ch, gg in g.groupby("characteristic"):
            ax.scatter(gg["modifier_slope"], gg["modifier_slope_u"], s=18, label=ch)
        lim = np.nanmax(np.abs(g[["modifier_slope", "modifier_slope_u"]].to_numpy())) * 1.1
        ax.plot([-lim, lim], [-lim, lim], ls="--", color="grey", lw=0.8)
        ax.axhline(0, color="0.8", lw=0.6); ax.axvline(0, color="0.8", lw=0.6)
        ax.set(xlabel="adjusted for distance + dilution", ylabel="not adjusted", title=f"{label}: change per SD ÷ threshold")
    axes[-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout(); plt.show()

## 10. Several characteristics together?

A model with several characteristics separates their influences, but each one needs data: we ask for **at least 10 dams per characteristic** with every value known, and characteristics that don't move together (|ρ| < 0.7, VIF < 5; VIF measures how much the overlap inflates uncertainty). The table says, per measure, whether that holds. With ~20–25 BACI dams, usually only one characteristic at a time is justified. The chart shows the combined model's results only where it is.

In [ ]:
if comb_check.empty:
    print("No combined model run (optional): COMBINED=\"a b\" RUNS=modifiers_combined sbatch data/slurm/fit_distance_models.slurm")
else:
    c = comb_check[comb_check["series"].isin(MAIN)].assign(design=lambda f: f["design"].map(DESIGNS))
    display(c[["series", "design", "characteristics", "complete_dams", "dams_per_modifier", "max_abs_rho",
               "max_vif", "justified", "reasons"]].fillna(""))
    ok = comb_check[comb_check["justified"].astype(bool)]
    terms = comb_coef[comb_coef["term"].str.match(r"(pd|post)_m_") & comb_coef["series"].isin(MAIN)] if len(comb_coef) else comb_coef
    terms = terms[terms["spec"].str.replace("_x_combined", "").isin(ok["design"]) & terms["series"].isin(ok["series"])
                  & ((terms["spec"].str.startswith("baci") & terms["term"].str.startswith("pd_"))
                     | (terms["spec"].str.startswith("before_after") & terms["term"].str.startswith("post_")))] if len(terms) else terms
    if len(terms):
        t = per_threshold(terms.assign(verdict=verdict(terms, threshold=False)), ["estimate", "ci_low", "ci_high"])
        t["label"] = t["series"] + " · " + t["spec"].str.replace("_x_combined", "").map(DESIGNS) + " · " + t["term"].str.replace(r"^(pd|post)_m_", "", regex=True)
        fig, ax = plt.subplots(figsize=(8, 0.35 * len(t) + 1))
        forest(ax, t, "estimate", "ci_low", "ci_high", "label", "verdict",
               title="Combined model: change per SD ÷ threshold", xlabel="change per SD ÷ threshold")
        fig.tight_layout(); plt.show()

## 11. Can we trust the fits?

One row per model that was skipped or flagged. **Skipped** = fewer than 10 dams. **fit check = check** = the optimizer reported an unstable solution (usually a random effect estimated at 0, harmless for the effect, but compare with the one-effect model). **Random effects** says whether each dam got its own effect (*slopes*) or not (*intercepts*), chosen by the data as in the main models.

In [ ]:
frames = [f.assign(run=run) for run, f in (("time", time_results), ("time_pretrend", pre_results),
                                            ("modifiers", mod_results), ("modifiers_combined", comb_results)) if len(f)]
if not frames:
    print("No model runs yet.")
else:
    allr = pd.concat(frames, ignore_index=True)
    allr = allr[allr["series"].isin(MAIN)]
    err = allr["error"].fillna("")
    singular = allr["singular"].fillna(False).astype(bool) if "singular" in allr else False
    flagged = allr[err.ne("") | allr["fit_check"].eq("check") | singular]
    print(f"{len(allr)} models: {int(err.eq('').sum())} fit, {int(err.ne('').sum())} skipped or failed, "
          f"{int(allr['fit_check'].eq('check').sum())} with fit check = check")
    if "re_choice" in allr:
        print("Random effects chosen:", allr.loc[err.eq(""), "re_choice"].str.split(" ").str[0].value_counts().to_dict())
    display(flagged[["run", "series", "spec", "dams", "re_choice", "fit_check", "near_zero_terms", "error"]].fillna(""))

**Notes:** time since removal is in calendar years (removal dates are known only to the year), so "year 1" spans 1–24 months after removal, and later intervals rest on fewer, partly different dams. Each dam may have its own overall effect, but the change over time is shared. The reservoir-size index comes from dam height, slope and river size; it is not a measured volume or sediment amount. Land cover is the downstream station's whole upstream watershed, which is larger than the dam's. A characteristic that goes with bigger effects is an association across dams, not proof it causes them. Full tables: `stage5/<run>__<random effects>/`; design notes: `docs/time_and_modifier_models.md`.